# Adversarial Attacks & Defences: CNN (ResNet-20) vs. Vision Transformer (ViT) on CIFAR-10

One-click notebook: **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

Pipeline: pretrained clean CNN/ViT -> native PyTorch FGSM & PGD -> 10-epoch AMP adversarial fine-tuning
(eps = 8/255) -> accuracy-vs-eps curves, cost of adversarial training, transferability, example grid.
Estimated total time on a T4: roughly 20-45 minutes (not yet measured; see README).

**Optional GitHub push:** add a Colab secret named `GITHUB_TOKEN` (fine-grained token, *Contents: read/write*
on the repo only). With it, checkpoints, CSV/MD tables and PNG plots are pushed after every stage and every 5
training epochs. Without it everything still runs, nothing is pushed.

In [ ]:
import base64, os, subprocess  # base64 = text encoding; os = files and folders; subprocess = run terminal commands

REPO_URL = "https://github.com/Mehtab404/DeepLearningProject.git"  # the GitHub project this notebook uses
MOUNT_DRIVE = True  # Colab only: keep saved models on Google Drive so a disconnect loses nothing
RUN_QUICK_TEST = False  # True = run a tiny test first (a few minutes on a GPU)
AUTO_PUSH = True  # upload results to GitHub; only works when a GITHUB_TOKEN secret is found
DRIVE_DIR = "/content/drive/MyDrive/adv_cnn_vs_vit_ckpts"  # Google Drive folder for the backups

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")  # first try: read the token (GitHub's secret password) from the computer's settings
if not GITHUB_TOKEN:  # not found there
    try:  # second try: Colab's secret storage
        from google.colab import userdata  # Colab's secret storage
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""  # read the secret (empty text if missing)
    except Exception:  # we are not on Colab
        pass  # ignore it
if not GITHUB_TOKEN:  # still not found
    try:  # third try: Kaggle's secret storage
        from kaggle_secrets import UserSecretsClient  # Kaggle's secret storage
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")  # read the secret
    except Exception:  # we are not on Kaggle
        pass  # ignore it

if GITHUB_TOKEN:  # a token was found
    basic = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()  # turn the token into the text format git's login needs
    os.environ.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",  # give the login to git through temporary settings (never saved in a file or log)
                       "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"})  # the login text itself
    print("GITHUB_TOKEN found: results will be pushed to GitHub.")  # tell the user
else:  # no token
    AUTO_PUSH = False  # switch uploading off
    print("No GITHUB_TOKEN found: running without GitHub pushes.")  # tell the user

BASE = "/content" if os.path.isdir("/content") else ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())  # the working folder: Colab's, else Kaggle's, else the current one
REPO_DIR = os.path.join(BASE, "DeepLearningProject")  # folder where the project will be placed
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):  # the project is not downloaded yet
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)  # download (clone) the project from GitHub
    if r.returncode != 0:  # the download failed
        print("Clone failed (private repo without token?). Continuing in a fresh local folder, pushes disabled.")  # tell the user
        os.makedirs(REPO_DIR, exist_ok=True)  # create an empty folder instead
        subprocess.run(["git", "init", "-b", "main"], cwd=REPO_DIR, capture_output=True)  # start a new empty git project there
        AUTO_PUSH = False  # switch uploading off
os.chdir(REPO_DIR)  # move into the project folder
subprocess.run(["git", "config", "user.name", "colab-agent"])  # name shown on the uploads
subprocess.run(["git", "config", "user.email", "colab-agent@users.noreply.github.com"])  # email shown on the uploads
os.makedirs("src", exist_ok=True); os.makedirs("checkpoints", exist_ok=True); os.makedirs("outputs", exist_ok=True)  # create the folders for code, saved models and results
print("Working in", os.getcwd())  # show the working folder

if MOUNT_DRIVE and os.path.isdir("/content"):  # on Colab and Drive is wanted
    try:  # attempt to connect Google Drive
        from google.colab import drive  # Colab's Drive tool
        drive.mount("/content/drive")  # connect Google Drive (asks for permission)
    except Exception as e:  # it did not work
        print("Drive mount skipped:", type(e).__name__)  # tell the user
        DRIVE_DIR = ""  # do not use Drive
else:  # not on Colab, or Drive not wanted
    DRIVE_DIR = ""  # do not use Drive

In [ ]:
# next line installs the extra libraries the project needs
!pip -q install timm transformers seaborn tqdm pandas
import torch  # main library for deep learning
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),  # show the library version and whether a GPU is available
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (use --quick-test)")  # show the GPU name, or a warning that only the CPU is available

In [ ]:
%%writefile src/__init__.py
# Adversarial robustness: CNN vs ViT on CIFAR-10  <- this folder is a Python package (a group of code files)

In [ ]:
%%writefile src/utils.py
"""Small helper tools used by the other files."""
import json  # read and write JSON text files
import os  # work with file and folder paths
import random  # Python's random numbers
import shutil  # copy files
import subprocess  # run terminal commands from Python
import sys  # access to the error output

import numpy as np  # fast number lists
import torch  # main library for deep learning

SEED = 42  # one fixed starting number for all randomness, so results can be repeated


def set_seed(seed: int = SEED):  # make all random numbers repeatable
    random.seed(seed)  # fix Python's random numbers
    np.random.seed(seed)  # fix NumPy's random numbers
    torch.manual_seed(seed)  # fix PyTorch's random numbers
    if torch.cuda.is_available():  # if a GPU exists
        torch.cuda.manual_seed_all(seed)  # fix the GPU's random numbers too


def get_device() -> torch.device:  # choose where to run the maths
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU (fast) if there is one, else CPU (slow)


def save_json(obj, path):  # save a Python dictionary or list as a readable text file
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)  # create the folder if it is missing
    with open(path, "w", encoding="utf-8") as f:  # open the file for writing
        json.dump(obj, f, indent=2)  # write the data neatly indented


def load_json(path, default=None):  # read a JSON file; if it does not exist, give back `default`
    if not os.path.exists(path):  # file is missing
        return default  # give the fallback value instead of crashing
    with open(path, "r", encoding="utf-8") as f:  # open the file for reading
        return json.load(f)  # read and return its content


def _exclude_oversized_files(repo_dir, limit_mb=95):  # tell git to ignore saved-model files that are too big for GitHub (limit about 100 MB)
    try:  # never crash because of this
        exclude = os.path.join(repo_dir, ".git", "info", "exclude")  # git's private "ignore" list (not shared)
        if not os.path.isdir(os.path.dirname(exclude)):  # not a git folder
            return  # nothing to do
        existing = open(exclude, encoding="utf-8").read() if os.path.exists(exclude) else ""  # read what is already in the ignore list
        new = []  # big files found
        for root, _dirs, files in os.walk(os.path.join(repo_dir, "checkpoints")):  # look through the saved-models folder
            for f in files:  # check every file
                p = os.path.join(root, f)  # full path of the file
                if f.endswith((".pth", ".pt")) and os.path.getsize(p) > limit_mb * 1024 * 1024:  # a saved-model file bigger than the limit
                    rel = os.path.relpath(p, repo_dir).replace(os.sep, "/")  # its path inside the project
                    if rel not in existing:  # not in the ignore list yet
                        new.append(rel)  # remember it
        if new:  # at least one big file found
            with open(exclude, "a", encoding="utf-8") as fh:  # open the ignore list to add lines
                fh.write("\n" + "\n".join(new) + "\n")  # add the big files
            print(f"[AUTO-PUSH] Excluded from git (>{limit_mb}MB): {', '.join(new)}")  # tell the user
    except OSError:  # a file problem happened
        pass  # ignore it quietly


def git_auto_push(message: str, repo_dir: str = ".") -> bool:  # saves the work to GitHub automatically; never crashes the training
    def run(*cmd):  # small helper that runs one terminal command
        return subprocess.run(cmd, cwd=repo_dir, capture_output=True, text=True)  # run it and collect its output instead of printing it

    try:  # attempt the upload
        if run("git", "rev-parse", "--is-inside-work-tree").returncode != 0:  # check this folder is a git project
            print("[AUTO-PUSH] Not a git repository, skipping.")  # it is not
            return False  # stop
        _exclude_oversized_files(repo_dir)  # skip files that are too big for GitHub
        run("git", "add", ".")  # mark all changed files to be saved
        if run("git", "diff", "--cached", "--quiet").returncode == 0:  # code 0 means nothing changed
            print("[AUTO-PUSH] No file changes to commit.")  # tell the user
            return True  # nothing to upload
        commit = run("git", "commit", "-m", message)  # save a snapshot with the given message
        if commit.returncode != 0:  # saving the snapshot failed
            print(f"[AUTO-PUSH] Commit failed: {commit.stderr.strip()[:300]}")  # show the reason
            return False  # stop
        push = run("git", "push", "origin", "HEAD")  # upload the snapshot to GitHub
        if push.returncode != 0:  # the upload failed
            print("[AUTO-PUSH] Push failed (check remote / token). Commit kept locally.")  # tell the user (the details are hidden because they may contain secrets)
            return False  # stop
        print(f"[AUTO-PUSH] Successfully pushed to GitHub: {message}")  # tell the user it worked
        return True  # success
    except Exception as e:  # noqa: BLE001 - anything unexpected
        print(f"[AUTO-PUSH] Skipped due to error: {type(e).__name__}")  # tell the user, but keep going
        return False  # stop


def sync_to_drive(src_dir: str, drive_dir: str):  # copy a folder's files to Google Drive, so nothing is lost if Colab disconnects
    if not drive_dir:  # no Drive folder given
        return  # do nothing
    try:  # attempt the copy
        os.makedirs(drive_dir, exist_ok=True)  # create the Drive folder if it is missing
        for name in os.listdir(src_dir):  # go through everything in the source folder
            s = os.path.join(src_dir, name)  # full path of the item
            if os.path.isfile(s):  # only copy files, not folders
                shutil.copy2(s, os.path.join(drive_dir, name))  # copy the file to Drive
    except Exception as e:  # noqa: BLE001 - copying failed
        print(f"[DRIVE] Sync skipped: {type(e).__name__}: {e}", file=sys.stderr)  # tell the user, but keep going


def restore_from_drive(dest_dir: str, drive_dir: str):  # the opposite: bring saved files back from Google Drive
    if not drive_dir or not os.path.isdir(drive_dir):  # no Drive folder to read from
        return  # do nothing
    os.makedirs(dest_dir, exist_ok=True)  # create the local folder if it is missing
    for name in os.listdir(drive_dir):  # go through the files on Drive
        d = os.path.join(dest_dir, name)  # where the file should go here
        s = os.path.join(drive_dir, name)  # where it is on Drive
        if os.path.isfile(s) and not os.path.exists(d):  # it is a file and we do not have it yet
            shutil.copy2(s, d)  # copy it here
            print(f"[DRIVE] Restored {name}")  # tell the user

In [ ]:
%%writefile src/models.py
"""Models: the two image-recognising models we compare (a CNN and a ViT)."""
import os  # work with file and folder paths
import time  # used to wait between retries

import torch  # main library for deep learning
import torch.nn as nn  # building blocks for models
import torch.nn.functional as F  # ready-made maths functions (we use it to resize images)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)  # average red, green, blue value of the training images
CIFAR_STD = (0.2470, 0.2435, 0.2616)  # how much red, green, blue values usually vary from that average

CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",  # the 10 picture types; label 0 = airplane, 1 = automobile, ...
                   "dog", "frog", "horse", "ship", "truck"]  # ... label 9 = truck

HF_VIT_ID = "aaraki/vit-base-patch16-224-in21k-finetuned-cifar10"  # name of an optional bigger ViT on the HuggingFace website
HUB_REPO = "chenyaofo/pytorch-cifar-models:master"  # GitHub place where the ready-made CNN is downloaded from


def _with_retries(fn, what="download", attempts=5, base_delay=5):  # runs fn(); if the internet fails, waits and tries again
    for i in range(1, attempts + 1):  # try up to 5 times
        try:  # attempt the job
            return fn()  # success: give back the result and stop
        except Exception as e:  # noqa: BLE001 - something went wrong (e.g. network problem)
            if i == attempts:  # this was the last try
                raise  # give up and show the error
            wait = base_delay * i  # wait longer after each failure: 5s, 10s, 15s ...
            print(f"[retry] {what} failed ({type(e).__name__}: {str(e)[:120]}); attempt {i}/{attempts}, "  # tell the user what failed
                  f"retrying in {wait}s")  # and how long we wait
            time.sleep(wait)  # pause before trying again


class NormalizedModelWrapper(nn.Module):  # a cover around a model: it resizes and rescales the image first, so outside the model images stay plain 0-to-1
    def __init__(self, model, mean=CIFAR_MEAN, std=CIFAR_STD, img_size=None):  # set up the cover
        super().__init__()  # start the basic PyTorch model setup
        self.model = model  # the real model inside (CNN or ViT)
        self.img_size = img_size  # size to enlarge images to (None = keep the size; 224 for the ViT)
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))  # store the average colour inside the model (fixed numbers, not learned)
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))  # store the colour variation inside the model (fixed numbers, not learned)

    def forward(self, x):  # runs when we give an image to the model
        if self.img_size is not None and x.shape[-1] != self.img_size:  # the ViT needs bigger images than the small 32x32 ones
            x = F.interpolate(x, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)  # enlarge the image smoothly to 224x224
        x_norm = (x - self.mean) / self.std  # subtract the average colour and divide by the variation (models learn better this way)
        out = self.model(x_norm)  # run the real model
        return out.logits if hasattr(out, 'logits') else out  # some models wrap their 10 scores in an object; take the scores out of it


def load_cnn_clean(device, pretrained=True, arch="cifar10_resnet20"):  # get the normally trained CNN (ResNet-20)
    base = _with_retries(lambda: torch.hub.load(HUB_REPO, arch, pretrained=pretrained, trust_repo=True),  # download the ready-made CNN (retry if the internet fails)
                         what="torch.hub " + arch)  # name used in the retry message
    model = NormalizedModelWrapper(base, CIFAR_MEAN, CIFAR_STD, img_size=None)  # put the cover around it; the CNN keeps 32x32 images
    return model.to(device).eval()  # move it to the GPU/CPU and switch to test mode


def load_vit_clean(device, backbone="timm-tiny", pretrained=True):  # get the ViT; also says whether it still needs normal training
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)  # the ViT expects colours centred at 0.5 with a spread of 0.5
    if backbone == "timm-tiny":  # option 1: small ViT (fast)
        import timm  # library that contains many ready-made models
        base = _with_retries(  # download the small ViT (retry if the internet fails)
            lambda: timm.create_model("vit_tiny_patch16_224", pretrained=pretrained, num_classes=10),  # 224x224 input cut into 16x16 pieces; 10 outputs for our 10 classes
            what="timm vit_tiny weights")  # name used in the retry message
        needs_ft = pretrained  # a downloaded ViT has a new empty 10-class end part, so it needs a short normal training
    elif backbone == "hf-base":  # option 2: bigger ViT from HuggingFace (slower)
        from transformers import AutoConfig, ViTForImageClassification  # tools to load that bigger ViT
        if pretrained:  # download the already-learned version
            base = _with_retries(lambda: ViTForImageClassification.from_pretrained(HF_VIT_ID),  # download it (retry if the internet fails)
                                 what="HF ViT-Base weights")  # name used in the retry message
        else:  # only build an empty model; saved weights will be loaded into it later
            base = ViTForImageClassification(AutoConfig.from_pretrained(HF_VIT_ID))  # same shape, random numbers
        needs_ft = False  # this one is already trained on CIFAR-10
    else:  # any other name is a mistake
        raise ValueError(f"Unknown ViT backbone: {backbone}")  # show an error
    model = NormalizedModelWrapper(base, mean, std, img_size=224)  # put the cover around it; images are enlarged to 224x224
    return model.to(device).eval(), needs_ft  # move to the GPU/CPU, test mode; also say if it needs normal training


def save_checkpoint(model, path):  # save the model's learned numbers (weights) to a file
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)  # create the folder if it is missing
    torch.save(model.state_dict(), path)  # write all learned numbers to the file


def load_checkpoint(model, path, device):  # load saved weights into a model of the same shape
    model.load_state_dict(torch.load(path, map_location=device))  # read the file and copy the numbers into the model (works on GPU or CPU)
    return model.to(device).eval()  # move to the GPU/CPU and switch to test mode


def sanity_check(device=None):  # quick check that both models take a 32x32 image and give 10 scores
    device = device or torch.device("cpu")  # use the CPU if no device was given
    x = torch.rand(4, 3, 32, 32, device=device)  # 4 fake colour images of size 32x32
    cnn = load_cnn_clean(device)  # load the CNN
    vit, _ = load_vit_clean(device)  # load the ViT (ignore the second answer)
    with torch.no_grad():  # no learning information needed for a simple test (saves memory)
        for name, m in (("CNN", cnn), ("ViT", vit)):  # test both models
            out = m(x)  # get the scores for the 4 fake images
            assert out.shape == (4, 10), f"{name} output shape {tuple(out.shape)} != (4, 10)"  # must be 4 images x 10 scores, else stop with an error
            print(f"[sanity] {name}: input {tuple(x.shape)} -> output {tuple(out.shape)} OK")  # tell the user it is fine


if __name__ == "__main__":  # true only when this file is run directly
    sanity_check()  # run the quick check

In [ ]:
%%writefile src/attacks.py
"""Attacks: small tricks that change an image a tiny bit so the model gets it wrong."""
import torch  # main library for deep learning
import torch.nn.functional as F  # ready-made maths functions (we use it for the loss)


def _input_grad(model, x, y, amp=False):  # finds which way to change each pixel so the model becomes MORE wrong
    x = x.clone().detach().requires_grad_(True)  # copy the image and tell PyTorch we want the push direction for the IMAGE (not the weights)
    use_amp = bool(amp) and x.is_cuda  # faster half-precision maths only if asked AND a GPU is used
    with torch.autocast(device_type="cuda" if x.is_cuda else "cpu", enabled=use_amp):  # switch on the faster maths when allowed
        logits = model(x)  # model gives 10 scores per image (one score for each class)
    loss = F.cross_entropy(logits.float(), y, reduction="sum")  # loss = how wrong the model is; "sum" adds it over all images
    grad = torch.autograd.grad(loss, x)[0]  # for every pixel: does raising it make the loss go up or down?
    return torch.nan_to_num(grad.detach())  # replace any broken (NaN) numbers with 0 so they cannot ruin the image


def fgsm_attack(model, x, y, eps, amp=False):  # FGSM attack: ONE quick step that pushes every pixel to make the model wrong
    if eps == 0:  # eps = biggest change allowed per pixel; 0 means "do not attack"
        return x.clone().detach()  # give back an untouched copy of the image
    grad = _input_grad(model, x, y, amp)  # get the direction that makes the model more wrong
    x_adv = x.detach() + eps * grad.sign()  # move every pixel by exactly +eps or -eps (sign keeps only the + or - direction)
    return torch.clamp(x_adv, 0.0, 1.0).detach()  # keep pixel values between 0 (black) and 1 (white)


def pgd_attack(model, x, y, eps, alpha=None, steps=10, random_start=True, amp=False):  # PGD attack: many small steps, so much stronger than FGSM
    if eps == 0:  # no change allowed
        return x.clone().detach()  # return the image as it is
    if alpha is None:  # alpha = size of ONE small step
        alpha = max(eps / 4.0, 1.0 / 255.0)  # default step = a quarter of eps (but never below 1/255)
    x = x.detach()  # the original image; it is never changed
    if random_start:  # start from a slightly shaken image instead of the exact original
        x_adv = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0.0, 1.0)  # add random noise between -eps and +eps, keep pixels in 0..1
    else:  # no random start
        x_adv = x.clone()  # start from an exact copy of the original
    for _ in range(steps):  # repeat the small step several times
        grad = _input_grad(model, x_adv, y, amp)  # direction that makes the model more wrong, for the current tricked image
        x_adv = x_adv + alpha * grad.sign()  # take one small step in that direction
        delta = torch.clamp(x_adv - x, -eps, eps)  # total change so far, cut off so no pixel moved more than eps
        x_adv = torch.clamp(x + delta, 0.0, 1.0)  # original + allowed change, pixels kept between 0 and 1
    return x_adv.detach()  # the final tricked image


def rs_fgsm_attack(model, x, y, eps, alpha=None, amp=False):  # RS-FGSM: random start + ONE step; fast, used while TRAINING the defence
    if eps == 0:  # no change allowed
        return x.clone().detach()  # return the image as it is
    if alpha is None:  # alpha = size of the one step
        alpha = 1.25 * eps  # default step is a bit bigger than eps (the paper's advice)
    x = x.detach()  # the original image; it is never changed
    x0 = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0.0, 1.0)  # random start: add random noise, keep pixels in 0..1
    grad = _input_grad(model, x0, y, amp)  # direction that makes the model more wrong, from the random start
    delta = torch.clamp(x0 + alpha * grad.sign() - x, -eps, eps)  # one step, then cut the total change to within +-eps
    return torch.clamp(x + delta, 0.0, 1.0).detach()  # original + allowed change, pixels kept between 0 and 1


def check_perturbation(x, x_adv, eps, tol=1e-6):  # safety check: did the attack stay within the rules?
    linf = torch.max(torch.abs(x_adv - x)).item()  # the biggest change made to any single pixel
    assert linf <= eps + tol, f"L-inf violated: {linf:.8f} > eps {eps:.8f}"  # stop with an error if some pixel changed more than eps (tol = tiny rounding allowance)
    assert 0.0 <= x_adv.min().item() and x_adv.max().item() <= 1.0, "x_adv left the [0, 1] range"  # stop with an error if any pixel is outside 0..1
    return linf  # report the biggest change


def self_test(device=None, n=8):  # quick built-in test with fake images; run it with: python -m src.attacks
    import torch.nn as nn  # building blocks for a tiny model
    device = device or torch.device("cpu")  # use the CPU if no device was given
    torch.manual_seed(0)  # fixed starting number so the random numbers are the same every time
    model = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),  # tiny test model: look at the image, keep positive signals, average them
                          nn.Flatten(), nn.Linear(8, 10)).to(device).eval()  # then flatten and give 10 class scores; eval = test mode
    x = torch.rand(n, 3, 32, 32, device=device)  # n fake colour images of size 32x32
    y = torch.randint(0, 10, (n,), device=device)  # n fake labels between 0 and 9
    for eps in (0.0, 1 / 255, 4 / 255, 8 / 255, 16 / 255):  # try several attack strengths
        for name, fn in (("fgsm", lambda: fgsm_attack(model, x, y, eps)),  # test the FGSM attack
                         ("pgd10", lambda: pgd_attack(model, x, y, eps, steps=10)),  # test PGD with 10 steps
                         ("pgd-no-rs", lambda: pgd_attack(model, x, y, eps, steps=3, random_start=False)),  # test PGD with 3 steps and no random start
                         ("rs_fgsm", lambda: rs_fgsm_attack(model, x, y, eps))):  # test RS-FGSM
            x_adv = fn()  # run the attack
            linf = check_perturbation(x, x_adv, eps)  # make sure it followed the rules
            if eps == 0:  # with eps = 0 nothing should change
                assert linf == 0.0, f"{name}: eps=0 must return x unchanged"  # stop with an error if the image changed
        print(f"[attacks] eps={eps * 255:.0f}/255: fgsm / pgd / rs_fgsm all satisfy L-inf <= eps and [0, 1]")  # tell the user this strength passed
    assert all(p.grad is None for p in model.parameters()), "attacks must not accumulate parameter grads"  # attacks must not leave any training information inside the model
    print("[attacks] self-test passed")  # all checks passed


if __name__ == "__main__":  # true only when this file is run directly
    self_test()  # run the quick test

In [ ]:
%%writefile src/data.py
"""Data: get the CIFAR-10 pictures (60,000 small pictures of 10 kinds of things) ready."""
import os  # work with file and folder paths
import shutil  # copy files
import time  # wait between retries

import numpy as np  # fast number lists
import torch  # main library for deep learning
from torch.utils.data import DataLoader, Subset  # DataLoader hands out images in groups; Subset keeps only some images
from torchvision import datasets, transforms  # ready-made picture datasets and picture changes

from .utils import SEED  # the fixed starting number for randomness

CIFAR_TAR = "cifar-10-python.tar.gz"  # name of the downloaded dataset file


def prepare_data(root="data", drive_dir=None, attempts=3):  # make sure the dataset is on the computer (download it if not)
    dst = os.path.join(root, CIFAR_TAR)  # where the dataset file should be on this computer
    cached = os.path.join(drive_dir, "dataset", CIFAR_TAR) if drive_dir else None  # backup copy on Google Drive (only if a Drive folder is given)
    os.makedirs(root, exist_ok=True)  # create the data folder if it is missing
    if cached and not os.path.exists(dst) and os.path.exists(cached):  # we have a Drive copy but no local copy
        shutil.copy2(cached, dst)  # copy it here (much faster than downloading)
        print("[data] restored CIFAR-10 archive from Drive cache")  # tell the user
    for i in range(1, attempts + 1):  # try the download up to 3 times
        try:  # attempt the download
            datasets.CIFAR10(root=root, train=True, download=True)  # get the 50,000 training pictures (skipped if already there)
            datasets.CIFAR10(root=root, train=False, download=True)  # get the 10,000 test pictures (skipped if already there)
            break  # worked, so stop trying
        except Exception as e:  # noqa: BLE001 - the download failed
            if i == attempts:  # this was the last try
                raise  # give up and show the error
            print(f"[retry] CIFAR-10 download failed ({type(e).__name__}); attempt {i}/{attempts}")  # tell the user it will retry
            time.sleep(10 * i)  # wait longer each time (10s, 20s, ...)
    if cached and os.path.exists(dst) and not os.path.exists(cached):  # we have a local copy but no Drive copy yet
        os.makedirs(os.path.dirname(cached), exist_ok=True)  # create the Drive folder
        shutil.copy2(dst, cached)  # save a copy to Drive for next time
        print("[data] cached CIFAR-10 archive to Drive")  # tell the user


def get_train_loader(batch_size=128, root="data", max_samples=0, augment=True, num_workers=2, seed=SEED):  # gives training pictures in shuffled groups (batches)
    tfms = []  # list of changes to apply to each picture
    if augment:  # augment = make small random changes so the model sees more variety
        tfms += [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]  # slightly shift the picture; sometimes mirror it left-right
    tfms.append(transforms.ToTensor())  # turn the picture into numbers between 0 and 1 (no other rescaling here)
    ds = datasets.CIFAR10(root=root, train=True, download=True, transform=transforms.Compose(tfms))  # the training dataset with those changes
    if max_samples and max_samples < len(ds):  # use only part of the data (for quick tests)
        g = np.random.RandomState(seed)  # random generator with a fixed start so the same pictures are picked every time
        idx = g.permutation(len(ds))[:max_samples].tolist()  # shuffle all positions and keep the first max_samples
        ds = Subset(ds, idx)  # keep only those pictures
    pin = torch.cuda.is_available()  # True if a GPU exists (then copying data to it can be faster)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,  # hand out groups of batch_size pictures in a new random order each epoch
                      pin_memory=pin, drop_last=True,  # drop_last = skip the last small group so all groups have equal size
                      persistent_workers=num_workers > 0)  # keep the helper processes alive between epochs (faster)


def stratified_indices(targets, n, seed=SEED):  # picks the same number of test pictures from every class, so the test is fair
    targets = np.asarray(targets)  # list of labels as a number array
    classes = np.unique(targets)  # the different classes: 0 to 9
    per_class = max(1, n // len(classes))  # pictures per class, e.g. 1000 // 10 = 100
    rng = np.random.RandomState(seed)  # random generator with a fixed start
    idx = []  # the chosen picture positions
    for c in classes:  # go through each class
        cls_idx = np.where(targets == c)[0]  # positions of all pictures of this class
        idx.extend(rng.choice(cls_idx, size=min(per_class, len(cls_idx)), replace=False).tolist())  # pick per_class of them at random, never the same one twice
    return sorted(idx)  # give back the positions in order


def get_eval_tensors(n=1000, root="data", seed=SEED):  # builds the fixed test set used to compare all models
    ds = datasets.CIFAR10(root=root, train=False, download=True, transform=transforms.ToTensor())  # the test pictures as numbers between 0 and 1 (no random changes)
    idx = stratified_indices(ds.targets, n, seed)  # choose an equal number of pictures from each class
    X = torch.stack([ds[i][0] for i in idx])  # stack the chosen pictures into one big block of numbers
    Y = torch.tensor([ds.targets[i] for i in idx], dtype=torch.long)  # their true labels as whole numbers
    return X, Y  # pictures and labels

In [ ]:
%%writefile src/defense.py
"""Defence: train the model on tricked images too, so it learns not to be fooled."""
import os  # work with file and folder paths
import time  # measure how long things take

import torch  # main library for deep learning
import torch.nn.functional as F  # ready-made maths functions (we use it for the loss)
from tqdm import tqdm  # shows a progress bar while training

from .attacks import pgd_attack, rs_fgsm_attack  # the two attacks we use to make tricked images during training
from .models import save_checkpoint  # function that saves the model's learned numbers to a file
from .utils import git_auto_push, load_json, save_json, sync_to_drive  # helper functions from utils.py


def _autocast(device, enabled):  # switches on faster half-precision maths (only on a GPU)
    return torch.autocast(device_type=device.type, enabled=bool(enabled) and device.type == "cuda")  # on only if asked AND a GPU is used


def make_train_attack(method, eps):  # picks which attack makes the tricked images during training
    if method == "pgd5":  # option 1: PGD with 5 small steps (stronger, slower)
        return lambda m, x, y: pgd_attack(m, x, y, eps=eps, alpha=2 / 255, steps=5, random_start=True, amp=True)  # 5 steps of size 2/255 with a random start
    if method == "rs_fgsm":  # option 2: RS-FGSM, one step (faster, a bit less stable)
        return lambda m, x, y: rs_fgsm_attack(m, x, y, eps=eps, alpha=1.25 * eps, amp=True)  # one step of size 1.25 * eps with a random start
    raise ValueError(f"method must be 'pgd5' or 'rs_fgsm', got {method!r}")  # any other name is a mistake, so show an error


def update_history(history_path, tag, entry=None, **meta):  # keeps a log file of how training went (loss and accuracy per epoch)
    hist = load_json(history_path, {}) or {}  # read the old log (or start an empty one)
    rec = hist.setdefault(tag, {"epochs": []})  # find this model's part of the log (or create it)
    rec.update(meta)  # store extra settings such as method, eps, lr
    if entry is not None:  # an entry = the result of one finished epoch
        rec["epochs"] = [e for e in rec["epochs"] if e["epoch"] != entry["epoch"]] + [entry]  # replace an old entry of the same epoch, then add the new one
    save_json(hist, history_path)  # write the log back to the file


def is_training_complete(history_path, tag, epochs):  # True if this model has already finished all its epochs
    rec = (load_json(history_path, {}) or {}).get(tag, {})  # read this model's part of the log
    return bool(rec.get("completed")) and len(rec.get("epochs", [])) >= epochs  # marked finished AND enough epochs are recorded


def adversarial_finetune(model, train_loader, epochs=10, eps=8 / 255, method="pgd5", lr=1e-4,  # train an already-trained model more, using tricked images too
                         device=None, tag="cnn_adv", ckpt_dir="checkpoints", drive_dir=None,  # device = GPU/CPU, tag = name, ckpt_dir = folder for saved files
                         auto_push_every=0, push_fn=git_auto_push, max_batches=0):  # optional GitHub upload; max_batches only limits the quick test
    device = device or next(model.parameters()).device  # if no device was given, use the one the model is already on
    model.to(device)  # move the model to the GPU/CPU
    attack = make_train_attack(method, eps)  # choose the attack used to make tricked images
    use_amp = device.type == "cuda"  # use faster half-precision maths only on a GPU
    ckpt_path = os.path.join(ckpt_dir, f"{tag}.pth")  # file for the final learned numbers (weights)
    state_path = os.path.join(ckpt_dir, f"{tag}_state.pt")  # file with everything needed to continue after a crash
    hist_path = os.path.join(ckpt_dir, "training_history.json")  # the log file of loss and accuracy
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)  # the tool that updates the weights; lr = how big each update is; weight_decay stops weights growing too large
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # slowly lowers the learning rate towards 0 as epochs pass
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)  # stops very small numbers becoming 0 in half-precision maths
    start_epoch = 0  # begin from the first epoch unless we find a saved state
    if os.path.exists(state_path):  # a saved state means an earlier run stopped half-way
        st = torch.load(state_path, map_location=device)  # load that saved state
        model.load_state_dict(st["model"])  # put the saved weights back into the model
        optimizer.load_state_dict(st["optimizer"])  # restore the weight-updating tool
        scheduler.load_state_dict(st["scheduler"])  # restore the learning-rate plan
        scaler.load_state_dict(st["scaler"])  # restore the small-number protector
        start_epoch = st["epoch"]  # continue from the epoch where it stopped
        print(f"[{tag}] Resuming from epoch {start_epoch}/{epochs}")  # tell the user training is continuing
    update_history(hist_path, tag, completed=False, method=method, eps=eps, lr=lr, target_epochs=epochs,  # write the settings into the log and mark "not finished yet"
                   amp=use_amp, device=str(device))  # also store whether half-precision maths and which device are used
    for epoch in range(start_epoch, epochs):  # one loop = one epoch = one full pass over all training images
        t0 = time.time()  # note the start time of this epoch
        tot_loss = tot_clean = tot_adv = n = 0  # running totals: loss, correct normal guesses, correct tricked guesses, images seen
        pbar = tqdm(train_loader, desc=f"[{tag}] Epoch {epoch + 1}/{epochs}", leave=False)  # progress bar over the batches
        for b, (x, y) in enumerate(pbar):  # take one batch (a small group) of images x and their labels y
            if max_batches and b >= max_batches:  # only the quick test sets a limit
                break  # stop this epoch early
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)  # move the batch to the GPU/CPU
            model.eval()  # test mode while making tricked images (so the model does not change)
            with _autocast(device, use_amp):  # faster maths if a GPU is used
                x_adv = attack(model, x, y)  # make the tricked version of every image in the batch
            model.train()  # back to learning mode
            optimizer.zero_grad(set_to_none=True)  # erase the update directions left from the previous batch
            with _autocast(device, use_amp):  # faster maths if a GPU is used
                logits = model(torch.cat([x, x_adv], dim=0))  # give normal and tricked images together in one go; get 10 scores per image
            logits = logits.float()  # use normal-precision numbers for the loss
            bs = x.size(0)  # bs = number of images in the batch
            loss = 0.5 * F.cross_entropy(logits[:bs], y) + 0.5 * F.cross_entropy(logits[bs:], y)  # loss = half "mistakes on normal images" + half "mistakes on tricked images"
            scaler.scale(loss).backward()  # work out how every weight should change (made bigger first so small numbers survive)
            scaler.step(optimizer)  # actually change the weights
            scaler.update()  # adjust the protector for the next batch
            tot_loss += loss.item() * bs  # add this batch's loss to the running total
            tot_clean += (logits[:bs].argmax(1) == y).sum().item()  # count normal images guessed right (highest score = the guess)
            tot_adv += (logits[bs:].argmax(1) == y).sum().item()  # count tricked images guessed right
            n += bs  # count the images seen so far
            pbar.set_postfix(loss=f"{tot_loss / n:.3f}", clean=f"{100 * tot_clean / n:.1f}%",  # show the loss and the normal accuracy on the progress bar
                             robust=f"{100 * tot_adv / n:.1f}%")  # and the accuracy on tricked images
        pbar.close()  # close the progress bar
        scheduler.step()  # lower the learning rate a little, once per epoch
        model.eval()  # test mode again
        entry = {"epoch": epoch + 1, "train_loss": tot_loss / max(n, 1),  # summary of this epoch: epoch number and average loss (max(n, 1) avoids dividing by 0)
                 "train_clean_acc": 100 * tot_clean / max(n, 1), "train_robust_acc": 100 * tot_adv / max(n, 1),  # accuracy on normal images and on tricked images
                 "lr": optimizer.param_groups[0]["lr"], "seconds": time.time() - t0}  # learning rate used and time taken
        print(f"[{tag}] Epoch {entry['epoch']}/{epochs} | loss {entry['train_loss']:.4f} | "  # print the epoch result: epoch number and loss
              f"clean {entry['train_clean_acc']:.2f}% | robust {entry['train_robust_acc']:.2f}% | "  # normal accuracy and tricked-image accuracy
              f"{entry['seconds']:.0f}s")  # seconds taken
        save_checkpoint(model, ckpt_path)  # save the learned weights after every epoch
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),  # also save everything needed to continue later: weights, weight-updating tool
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),  # learning-rate plan, small-number protector
                    "epoch": epoch + 1}, state_path)  # and the epoch number
        update_history(hist_path, tag, entry=entry)  # add this epoch's result to the log
        sync_to_drive(ckpt_dir, drive_dir)  # copy the saved files to Google Drive as a backup (only if a Drive folder is given)
        if auto_push_every and (epoch + 1) % auto_push_every == 0 and (epoch + 1) < epochs:  # every few epochs, if upload is switched on
            push_fn(f"chore(train): {tag} epoch {epoch + 1}/{epochs} checkpoint")  # upload progress to GitHub
    update_history(hist_path, tag, completed=True)  # mark in the log that training is finished
    if os.path.exists(state_path):  # the "continue later" file is not needed any more
        os.remove(state_path)  # delete it
    sync_to_drive(ckpt_dir, drive_dir)  # final backup to Google Drive
    model.eval()  # leave the model in test mode
    return load_json(hist_path)[tag]["epochs"]  # give back the list of epoch results


def clean_finetune(model, train_loader, epochs=5, lr=2e-4, device=None, max_batches=0, desc="ViT-clean"):  # normal training (no tricked images); used once to prepare the ViT for CIFAR-10
    device = device or next(model.parameters()).device  # if no device was given, use the one the model is already on
    use_amp = device.type == "cuda"  # use faster half-precision maths only on a GPU
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)  # the tool that updates the weights
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # slowly lowers the learning rate towards 0
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)  # stops very small numbers becoming 0 in half-precision maths
    for epoch in range(epochs):  # one loop = one full pass over all training images
        model.train()  # learning mode
        tot = correct = n = 0  # running totals: loss, correct guesses, images seen
        pbar = tqdm(train_loader, desc=f"[{desc}] Epoch {epoch + 1}/{epochs}", leave=False)  # progress bar over the batches
        for b, (x, y) in enumerate(pbar):  # take one batch of images x and labels y
            if max_batches and b >= max_batches:  # only the quick test sets a limit
                break  # stop this epoch early
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)  # move the batch to the GPU/CPU
            optimizer.zero_grad(set_to_none=True)  # erase the update directions from the previous batch
            with _autocast(device, use_amp):  # faster maths if a GPU is used
                logits = model(x)  # 10 scores per image, normal images only
            loss = F.cross_entropy(logits.float(), y)  # loss = how wrong the guesses are
            scaler.scale(loss).backward()  # work out how every weight should change
            scaler.step(optimizer)  # change the weights
            scaler.update()  # adjust the small-number protector
            tot += loss.item() * x.size(0)  # add this batch's loss to the running total
            correct += (logits.argmax(1) == y).sum().item()  # count right guesses (highest score = the guess)
            n += x.size(0)  # count the images seen so far
            pbar.set_postfix(loss=f"{tot / n:.3f}", acc=f"{100 * correct / n:.1f}%")  # show loss and accuracy on the progress bar
        pbar.close()  # close the progress bar
        scheduler.step()  # lower the learning rate a little
        print(f"[{desc}] Epoch {epoch + 1}/{epochs} | loss {tot / max(n, 1):.4f} | acc {100 * correct / max(n, 1):.2f}%")  # print this epoch's loss and accuracy
    return model.eval()  # give back the trained model in test mode

In [ ]:
%%writefile src/evaluate.py
"""Evaluation: measure how accurate each model is, with and without attacks."""
import os  # work with file and folder paths

import pandas as pd  # tables (like Excel) in Python
import torch  # main library for deep learning

from .attacks import fgsm_attack, pgd_attack  # the two attacks used for testing

EPS_LIST = [0, 1, 2, 4, 8, 16]  # attack strengths to test, in 1/255 units (0 = no attack = normal accuracy)
MODEL_ORDER = ["CNN-Clean", "ViT-Clean", "CNN-AdvTrained", "ViT-AdvTrained"]  # the 4 models, in the order we show them


@torch.no_grad()  # do not keep learning information here (faster, uses less memory)
def _correct(model, x, y):  # counts how many images the model guesses right
    return (model(x).argmax(1) == y).sum().item()  # the highest of the 10 scores is the guess; count guesses equal to the true label


def accuracy(model, X, Y, device, batch_size=100):  # accuracy in % of the model on images X with true labels Y
    model.eval()  # test mode
    c = 0  # number of right guesses so far
    for i in range(0, len(X), batch_size):  # go through the images in groups of 100 (so memory does not run out)
        x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)  # take one group and move it to the GPU/CPU
        c += _correct(model, x, y)  # add the right guesses of this group
    return 100.0 * c / len(X)  # right guesses divided by total images, as a percentage


def _attack_fn(name, eps, pgd_steps):  # gives back the attack function with the test settings filled in
    if name == "FGSM":  # the one-step attack
        return lambda m, x, y: fgsm_attack(m, x, y, eps)  # attack with strength eps
    if name == "PGD-10":  # the strong many-step attack
        return lambda m, x, y: pgd_attack(m, x, y, eps, alpha=max(eps / 4, 1 / 255), steps=pgd_steps,  # small steps of size eps/4 (at least 1/255), repeated pgd_steps times
                                          random_start=True)  # begin from a randomly shaken image
    raise ValueError(name)  # any other name is a mistake, so show an error


def evaluate_epsilon_sweep(models, X, Y, device, eps_list=EPS_LIST, pgd_steps=10, batch_size=100, seed=42):  # test every model against every attack at every strength
    rows = []  # one row of results for each combination
    for name, model in models.items():  # go through the 4 models
        model.eval()  # test mode
        for attack in ("FGSM", "PGD-10"):  # go through both attacks
            for e in eps_list:  # go through every attack strength
                torch.manual_seed(seed)  # fixed starting number so the random parts are the same every run
                eps = e / 255.0  # change e.g. 8 into 8/255, because pixel values are between 0 and 1
                fn = _attack_fn(attack, eps, pgd_steps)  # get the attack function for this attack and strength
                correct = 0  # right guesses so far
                for i in range(0, len(X), batch_size):  # go through the test images in groups
                    x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)  # take one group and move it to the GPU/CPU
                    x_adv = fn(model, x, y)  # step 1: trick the images (the attacker uses this same model)
                    correct += _correct(model, x_adv, y)  # step 2: count how many tricked images are still guessed right
                acc = 100.0 * correct / len(X)  # accuracy in percent
                rows.append({"model": name, "attack": attack, "eps_255": e, "accuracy": acc})  # save this result as one row
                print(f"[eval] {name:15s} {attack:6s} eps={e:2d}/255 -> {acc:5.1f}%")  # show the result on screen
    return pd.DataFrame(rows)  # turn all rows into one table


def evaluate_transferability(models, X, Y, device, eps=8 / 255, pgd_steps=10, batch_size=100, seed=42):  # does a trick made on one model also fool another model?
    names = list(models.keys())  # the 4 model names
    mat = pd.DataFrame(index=names, columns=names, dtype=float)  # empty 4x4 table: rows = model that made the trick, columns = model being tested
    for src in names:  # take each model in turn as the one that makes the tricked images
        torch.manual_seed(seed)  # fixed starting number for repeatable results
        models[src].eval()  # test mode
        adv_batches = []  # will hold the tricked images
        for i in range(0, len(X), batch_size):  # go through the test images in groups
            x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)  # take one group and move it to the GPU/CPU
            adv_batches.append(pgd_attack(models[src], x, y, eps, alpha=max(eps / 4, 1 / 255),  # trick the group using the PGD attack on the source model
                                          steps=pgd_steps, random_start=True).cpu())  # many small steps, random start; move result to the CPU to save GPU memory
        X_adv = torch.cat(adv_batches)  # join all groups into one set of tricked images
        for tgt in names:  # test the same tricked images on every model
            mat.loc[src, tgt] = accuracy(models[tgt], X_adv, Y, device, batch_size)  # store that model's accuracy in the table
        print(f"[transfer] source={src:15s} -> " + ", ".join(f"{t}: {mat.loc[src, t]:.1f}%" for t in names))  # show this row of the table on screen
    mat.index.name = "source \\ target"  # label the table corner: rows = source, columns = target
    return mat  # give back the finished table


def to_markdown(df, floatfmt="{:.1f}"):  # turns a table into text that GitHub can show as a table
    cols = [df.index.name or ""] + [str(c) for c in df.columns]  # the column titles
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join([" --- "] * len(cols)) + "|"]  # title line plus the "---" divider line
    for idx, row in df.iterrows():  # go through each row of the table
        cells = [str(idx)] + [floatfmt.format(v) if isinstance(v, float) else str(v) for v in row.values]  # row name, then each number with 1 decimal place
        lines.append("| " + " | ".join(cells) + " |")  # add this row as a text line
    return "\n".join(lines)  # join all lines into one block of text


def export_results(sweep_df, transfer_df, out_dir, note=""):  # saves all results as files (CSV and Markdown)
    os.makedirs(out_dir, exist_ok=True)  # create the output folder if it is missing
    sweep_df.to_csv(os.path.join(out_dir, "results_summary.csv"), index=False)  # save the long results list as a CSV file
    wide = sweep_df.pivot_table(index=["attack", "model"], columns="eps_255", values="accuracy", sort=False)  # reshape: one row per attack+model, one column per strength
    wide.columns = [f"eps={c}/255" for c in wide.columns]  # give the columns readable names
    wide.index = [f"{a}, {m}" for a, m in wide.index]  # join attack and model into one row name
    wide.index.name = "attack, model"  # name of the row-title column
    md = ["# Results summary (accuracy %, white-box)", ""]  # first lines of the Markdown report
    if note:  # if a note was given
        md += [f"> {note}", ""]  # add it near the top
    md += [to_markdown(wide), "", "## Transferability: PGD-10 at eps=8/255 (rows = source, cols = target; accuracy %)", "",  # add the main table, then the heading of the second table
           to_markdown(transfer_df), ""]  # add the second table (which model's tricks fool which model)
    with open(os.path.join(out_dir, "results_summary.md"), "w", encoding="utf-8") as f:  # open the Markdown file for writing
        f.write("\n".join(md))  # write the report into it
    transfer_df.to_csv(os.path.join(out_dir, "transferability.csv"))  # save the second table as a CSV file
    print("\n".join(md))  # also show the report on screen

In [ ]:
%%writefile src/visualize.py
"""Pictures: draws the 4 result graphs and saves them as image files."""
import os  # work with file and folder paths

import matplotlib  # drawing library
matplotlib.use("Agg")  # draw into files only (no pop-up window), so it also works on a server
import matplotlib.pyplot as plt  # the drawing commands
import numpy as np  # fast number lists
import seaborn as sns  # nicer-looking graphs (we use it for the colour table)
import torch  # main library for deep learning

from .attacks import pgd_attack  # the strong attack, used to make the example pictures
from .models import CIFAR10_CLASSES  # names of the 10 picture types

DPI = 300  # picture sharpness (300 = print quality)
COLORS = {"CNN": "#1f77b4", "ViT": "#d62728"}  # blue for CNN, red for ViT


def _style(name):  # picks line colour, line style and marker shape for a model name
    arch = name.split("-")[0]  # first part of the name: "CNN" or "ViT"
    return COLORS[arch], ("--" if "Adv" in name else "-"), ("s" if "Adv" in name else "o")  # colour by type; dashed line + square for defended models, solid line + circle for normal ones


def _tag(fig, tag):  # writes a small grey note at the bottom of a graph
    if tag:  # only if a note was given (used to mark quick-test graphs as "not a real result")
        fig.text(0.5, 0.005, tag, ha="center", va="bottom", fontsize=8, color="gray")  # place the note in the bottom middle


def plot_acc_vs_epsilon(df, path, tag=""):  # graph 1: accuracy as the attack gets stronger (FGSM on the left, PGD-10 on the right)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)  # a canvas with two graphs side by side that share the up-down scale
    for ax, attack in zip(axes, ("FGSM", "PGD-10")):  # one graph for each attack
        for name in df["model"].unique():  # one line for each model
            d = df[(df.model == name) & (df.attack == attack)].sort_values("eps_255")  # keep this model's rows for this attack, ordered by strength
            c, ls, mk = _style(name)  # colour, line style and marker for this model
            ax.plot(d.eps_255, d.accuracy, color=c, linestyle=ls, marker=mk, linewidth=2, label=name)  # draw the line: attack strength across, accuracy up
        ax.set_title(f"{attack} attack")  # graph title
        ax.set_xlabel(r"$\epsilon$ (in /255 units)")  # label under the graph: attack strength
        ax.set_xticks(sorted(df.eps_255.unique()))  # show a mark for each strength we tested
        ax.grid(alpha=0.3)  # faint grid lines
    axes[0].set_ylabel("Accuracy (%)")  # label on the left side
    axes[0].set_ylim(0, 100)  # the up-down scale goes from 0 to 100 percent
    axes[1].legend(loc="upper right")  # box that says which line is which model
    fig.suptitle("Accuracy vs. perturbation budget on CIFAR-10 (solid = clean, dashed = adversarially fine-tuned)")  # main title
    fig.tight_layout(rect=(0, 0.03, 1, 1))  # fit everything neatly, leaving room at the bottom
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


def _acc(df, model, attack, eps):  # looks up one accuracy value in the results table
    return float(df[(df.model == model) & (df.attack == attack) & (df.eps_255 == eps)].accuracy.iloc[0])  # find the row for this model, attack and strength; take its accuracy


def plot_tradeoff(df, path, tag=""):  # graph 2: what defence training costs (normal accuracy) and gains (accuracy under attack)
    metrics = [("Clean (eps=0)", "FGSM", 0), ("FGSM (eps=8/255)", "FGSM", 8), ("PGD-10 (eps=8/255)", "PGD-10", 8)]  # three tests: no attack, FGSM, PGD-10 (FGSM with eps=0 is just normal accuracy)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)  # a canvas with two graphs side by side: CNN and ViT
    for ax, arch in zip(axes, ("CNN", "ViT")):  # one graph for each model type
        before = [_acc(df, f"{arch}-Clean", a, e) for _, a, e in metrics]  # accuracies of the normal model
        after = [_acc(df, f"{arch}-AdvTrained", a, e) for _, a, e in metrics]  # accuracies of the defended model
        xs = np.arange(len(metrics))  # positions of the bar groups: 0, 1, 2
        w = 0.36  # width of one bar
        ax.bar(xs - w / 2, before, w, label="Before (clean model)", color="#9ecae1" if arch == "CNN" else "#fcae91")  # light bars = before defence training
        ax.bar(xs + w / 2, after, w, label="After 10-epoch adv. fine-tuning", color=COLORS[arch])  # strong-colour bars = after defence training
        for i in range(len(metrics)):  # go through the three tests
            delta = after[i] - before[i]  # change in accuracy (in percentage points)
            ax.text(xs[i] - w / 2, before[i] + 1, f"{before[i]:.1f}", ha="center", fontsize=8)  # write the "before" number above its bar
            ax.text(xs[i] + w / 2, after[i] + 1, f"{after[i]:.1f}", ha="center", fontsize=8)  # write the "after" number above its bar
            label = r"$\Delta$Acc$_{clean}$" if i == 0 else r"$\Delta$Acc$_{robust}$"  # name of the change: normal-accuracy change or attacked-accuracy change
            ax.text(xs[i], max(before[i], after[i]) + 8, f"{label} = {delta:+.1f} pp", ha="center", fontsize=9,  # write the change above the bars
                    fontweight="bold", color="darkgreen" if delta >= 0 else "darkred")  # bold; green if accuracy went up, red if it went down
        ax.set_xticks(xs)  # one mark per bar group
        ax.set_xticklabels([m[0] for m in metrics])  # name each group
        ax.set_title(f"{arch}: cost of adversarial training vs. robustness gain")  # graph title
        ax.set_ylim(0, 115)  # the up-down scale (a bit above 100 to leave room for the text)
        ax.grid(axis="y", alpha=0.3)  # faint horizontal grid lines
        ax.legend(loc="upper right", fontsize=8)  # box that says which bar is which
    axes[0].set_ylabel("Accuracy (%)")  # label on the left side
    fig.tight_layout(rect=(0, 0.03, 1, 1))  # fit everything neatly
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


def plot_transferability(matrix, path, tag=""):  # graph 3: colour table showing which model's tricks fool which model
    fig, ax = plt.subplots(figsize=(7.5, 6))  # a canvas for one graph
    sns.heatmap(matrix.astype(float), annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100, ax=ax,  # coloured squares with the number written inside; red = low accuracy, green = high
                cbar_kws={"label": "Target accuracy (%) - lower = stronger attack"})  # label of the colour bar
    ax.set_xlabel("Target model (evaluated on)")  # label under the table: the model being tested
    ax.set_ylabel("Source model (adversarial examples crafted on)")  # label on the left: the model that made the tricked images
    ax.set_title(r"Transferability of PGD-10 examples ($\epsilon$ = 8/255); diagonal = white-box")  # title; the diagonal is where the same model is both attacked and tested
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")  # tilt the names under the table so they do not overlap
    fig.tight_layout(rect=(0, 0.03, 1, 1))  # fit everything neatly
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


def plot_adversarial_examples(models, X, Y, device, path, eps=8 / 255, source="CNN-Clean", n=4,  # graph 4: shows real pictures, the added noise, and the tricked pictures
                              pgd_steps=10, tag=""):  # pgd_steps = how many small steps the attack takes
    src = models[source].eval()  # the model that makes the tricked images, in test mode
    chosen, seen = [], set()  # chosen = picked picture positions; seen = classes already picked
    with torch.no_grad():  # no learning information needed here
        for i in range(len(X)):  # go through the test pictures
            y = int(Y[i])  # the true label of this picture
            if y in seen:  # we already have a picture of this class
                continue  # skip it
            if src(X[i:i + 1].to(device)).argmax(1).item() == y:  # the model guesses this picture right (an attack only makes sense then)
                chosen.append(i)  # pick it
                seen.add(y)  # remember this class is done
            if len(chosen) == n:  # we have enough pictures
                break  # stop looking
    if not chosen:  # nothing suitable was found
        chosen = list(range(min(n, len(X))))  # just take the first few pictures
    x = X[chosen].to(device)  # the chosen pictures, on the GPU/CPU
    y = Y[chosen].to(device)  # their true labels
    torch.manual_seed(42)  # fixed starting number so the random start is the same every time
    x_adv = pgd_attack(src, x, y, eps, alpha=max(eps / 4, 1 / 255), steps=pgd_steps, random_start=True)  # make the tricked version of each picture
    pert = ((x_adv - x + eps) / (2 * eps)).clamp(0, 1)  # the real noise is too small to see, so stretch it to the range 0..1 (grey = no change)
    preds = {}  # what each model guesses for the tricked pictures
    with torch.no_grad():  # no learning information needed here
        for name, m in models.items():  # go through all the models
            preds[name] = m.eval()(x_adv).argmax(1).cpu().tolist()  # the guess of this model for each tricked picture
    rows = len(chosen)  # one row of the graph per picture
    fig, axes = plt.subplots(rows, 3, figsize=(10.5, 3.1 * rows))  # a canvas with 3 columns: original, noise, tricked picture
    axes = np.atleast_2d(axes)  # makes sure indexing works even when there is only one row
    for r in range(rows):  # go through each picture
        axes[r, 0].imshow(x[r].permute(1, 2, 0).cpu().numpy())  # show the original picture (colour layers moved to the order the drawing tool wants)
        axes[r, 0].set_title(f"Clean: {CIFAR10_CLASSES[int(y[r])]}", fontsize=10)  # title: its true class name
        axes[r, 1].imshow(pert[r].permute(1, 2, 0).cpu().numpy())  # show the stretched noise
        axes[r, 1].set_title(r"PGD-10 perturbation, magnified $(x_{adv}-x+\epsilon)/2\epsilon$", fontsize=8)  # title: this is the magnified noise
        axes[r, 2].imshow(x_adv[r].permute(1, 2, 0).detach().cpu().numpy())  # show the tricked picture
        lines = []  # text lines with each model's guess
        for name in models:  # go through all the models
            p = preds[name][r]  # this model's guess
            ok = "OK" if p == int(y[r]) else "WRONG"  # is the guess right?
            lines.append(f"{name}: {CIFAR10_CLASSES[p]} [{ok}]")  # write: model name, guessed class, right or wrong
        axes[r, 2].set_title(f"x_adv (eps=8/255, crafted on {source})\n" + "\n".join(lines), fontsize=7)  # title: all models' guesses
        for c in range(3):  # go through the 3 columns
            axes[r, c].axis("off")  # hide the numbers around the picture
    fig.tight_layout(rect=(0, 0.02, 1, 1))  # fit everything neatly
    _tag(fig, tag)  # add the small grey note if there is one
    fig.savefig(path, dpi=DPI)  # save the graph as an image file
    plt.close(fig)  # release the memory


def make_all_plots(sweep_df, transfer_df, models, X, Y, device, out_dir, pgd_steps=10, tag=""):  # draws all four graphs and gives back their file paths
    os.makedirs(out_dir, exist_ok=True)  # create the output folder if it is missing
    paths = {  # file name for each graph
        "acc_vs_epsilon": os.path.join(out_dir, "acc_vs_epsilon.png"),  # graph 1
        "tradeoff": os.path.join(out_dir, "clean_vs_robust_tradeoff.png"),  # graph 2
        "transfer": os.path.join(out_dir, "transferability_heatmap.png"),  # graph 3
        "examples": os.path.join(out_dir, "adversarial_examples_grid.png"),  # graph 4
    }
    plot_acc_vs_epsilon(sweep_df, paths["acc_vs_epsilon"], tag)  # draw graph 1
    plot_tradeoff(sweep_df, paths["tradeoff"], tag)  # draw graph 2
    plot_transferability(transfer_df, paths["transfer"], tag)  # draw graph 3
    plot_adversarial_examples(models, X, Y, device, paths["examples"], pgd_steps=pgd_steps, tag=tag)  # draw graph 4
    for p in paths.values():  # go through the saved files
        print(f"[plots] saved {p}")  # tell the user where each one was saved
    return paths  # give back the file paths

In [ ]:
%%writefile main.py
"""CLI entry point: CNN vs ViT adversarial robustness on CIFAR-10.

  python main.py --quick-test            # CPU smoke test (64 test images, 1 tiny epoch), no GPU needed
  python main.py --full-run              # Colab/Kaggle GPU: 10-epoch adversarial fine-tuning + full evaluation
  python main.py --epochs 10 --eval-samples 1000 --auto-push

Checkpoints in checkpoints/ are reused automatically: nothing that already exists is retrained.
"""
import argparse  # read options typed in the terminal
import copy  # make independent copies of models
import json  # print / save results as JSON text
import os  # work with file and folder paths
import time  # measure how long the run takes

from src.data import get_eval_tensors, get_train_loader, prepare_data  # functions that get the pictures
from src.defense import adversarial_finetune, clean_finetune, is_training_complete  # functions that train the models
from src.evaluate import accuracy, evaluate_epsilon_sweep, evaluate_transferability, export_results  # functions that test the models and save results
from src.models import load_checkpoint, load_cnn_clean, load_vit_clean, save_checkpoint  # functions that create, load and save models
from src.utils import (SEED, get_device, git_auto_push, load_json, restore_from_drive, save_json, set_seed,  # small helper tools
                       sync_to_drive)  # backup to Google Drive
from src.visualize import make_all_plots  # function that draws the graphs


def parse_args():  # reads the options typed after "python main.py", e.g. --epochs 5
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)  # the option reader; shows the text at the top of this file when asked for help
    mode = p.add_mutually_exclusive_group()  # options in this group cannot be used together
    mode.add_argument("--quick-test", action="store_true", help="CPU smoke test: 64 images, 1 short epoch")  # tiny fast run just to check the code works
    mode.add_argument("--full-run", action="store_true", help="Full GPU run (default when no mode is given)")  # the real run
    p.add_argument("--epochs", type=int, default=10, help="adversarial fine-tuning epochs")  # how many times the model sees the whole training set
    p.add_argument("--eval-samples", type=int, default=1000, help="stratified test subset size")  # how many test images are used for testing
    p.add_argument("--eps", type=float, default=8.0, help="training epsilon in /255 units")  # biggest change per pixel used while training the defence
    p.add_argument("--cnn-method", choices=["pgd5", "rs_fgsm"], default="pgd5")  # which attack trains the CNN's defence
    p.add_argument("--vit-method", choices=["pgd5", "rs_fgsm"], default="pgd5",  # which attack trains the ViT's defence
                   help="pgd5 matches the CNN (fair comparison). rs_fgsm is ~2.5x faster but the ViT showed "  # help text, part 1
                        "signs of catastrophic overfitting with it (see outputs_vit_rsfgsm/)")  # help text, part 2
    p.add_argument("--vit-backbone", choices=["timm-tiny", "hf-base"], default="timm-tiny")  # which ViT to use: small (fast) or big (slow)
    p.add_argument("--vit-clean-epochs", type=int, default=5, help="clean fine-tune epochs for timm ViT-Tiny")  # epochs of normal training for the small ViT
    p.add_argument("--lr", type=float, default=1e-4)  # learning rate = how big each weight update is
    p.add_argument("--batch-size", type=int, default=128)  # how many images are processed together
    p.add_argument("--max-train-samples", type=int, default=0, help="0 = full 50k training set")  # use only part of the training images (0 = use all)
    p.add_argument("--num-workers", type=int, default=2)  # how many helper processes load images
    p.add_argument("--data-root", default="data")  # folder for the dataset
    p.add_argument("--ckpt-dir", default=None)  # folder for saved models
    p.add_argument("--out-dir", default="outputs")  # folder for result tables and graphs
    p.add_argument("--drive-dir", default=None, help="e.g. /content/drive/MyDrive/adv_ckpts (Colab)")  # Google Drive folder for backups
    p.add_argument("--force-retrain", action="store_true", help="ignore existing adversarial checkpoints")  # train again even if saved models exist
    p.add_argument("--auto-push", action="store_true", help="git commit+push after each stage / every 5 epochs")  # upload progress to GitHub automatically
    return p.parse_args()  # read the typed options and give them back


def clear_stale_quick_test_outputs(out_dir):  # deletes old quick-test results so fake numbers never mix with real ones
    """Quick-test artifacts are committed to git; delete them so they can never pass for full-run results."""
    info = load_json(os.path.join(out_dir, "run_info.json"), {}) or {}  # read the note that says which kind of run made the files
    if info.get("mode") != "quick-test":  # the files are from a real run (or missing)
        return  # keep them
    for name in ("acc_vs_epsilon.png", "clean_vs_robust_tradeoff.png", "transferability_heatmap.png",  # the files made by a quick test
                 "adversarial_examples_grid.png", "results_summary.csv", "results_summary.md",  # more of those files
                 "transferability.csv", "run_info.json"):  # and the last ones
        p = os.path.join(out_dir, name)  # full path of the file
        if os.path.exists(p):  # the file exists
            os.remove(p)  # delete it
    print("[outputs] removed stale quick-test artifacts")  # tell the user


def main():  # the main program: runs all the steps in order
    args = parse_args()  # read the typed options
    quick = args.quick_test  # True if this is the tiny test run
    device = get_device()  # GPU if there is one, else CPU
    set_seed(SEED)  # fix the random numbers so results can be repeated
    pgd_steps, max_batches, tag = 10, 0, ""  # normal settings: PGD uses 10 steps, no batch limit, no note on graphs
    if quick:  # tiny test run: make everything very small
        args.epochs, args.eval_samples, args.batch_size = 1, 64, 16  # 1 epoch, 64 test images, 16 images per batch
        args.max_train_samples, args.vit_clean_epochs, args.num_workers = 64, 1, 0  # 64 training images, 1 normal-training epoch, no helper processes
        args.drive_dir = None  # do not touch the real Google Drive backups
        pgd_steps, max_batches = 2, 2  # only 2 attack steps and 2 batches per epoch
        tag = "QUICK-TEST smoke run (tiny N, 1 epoch, 2 PGD steps) - not a result"  # note printed on the graphs
    args.ckpt_dir = args.ckpt_dir or ("checkpoints_quick_test" if quick else "checkpoints")  # saved models go to a separate folder for quick tests
    os.makedirs(args.ckpt_dir, exist_ok=True)  # create the saved-models folder
    os.makedirs(args.out_dir, exist_ok=True)  # create the results folder
    eps_train = args.eps / 255.0  # change e.g. 8 into 8/255, because pixel values are between 0 and 1

    print(f"Device: {device} | mode: {'quick-test' if quick else 'full-run'} | ckpt: {args.ckpt_dir}")  # show the setup
    if device.type == "cpu" and not quick:  # a full run without a GPU
        print("WARNING: full-run on CPU will be extremely slow. Use Colab/Kaggle GPU (or --quick-test).")  # warn the user

    restore_from_drive(args.ckpt_dir, args.drive_dir)  # bring back saved models from Google Drive (if used)
    if not quick:  # real run
        clear_stale_quick_test_outputs(args.out_dir)  # remove old quick-test results
    push = (lambda msg: git_auto_push(msg)) if args.auto_push else (lambda msg: None)  # a function that uploads to GitHub, or does nothing if auto-push is off
    t_start = time.time()  # note the start time

    cnn_path = os.path.join(args.ckpt_dir, "cnn_clean.pth")  # file of the normal CNN's weights
    vit_path = os.path.join(args.ckpt_dir, "vit_clean.pth")  # file of the normal ViT's weights

    if os.path.exists(cnn_path):  # the CNN was already saved earlier
        cnn_clean = load_checkpoint(load_cnn_clean(device, pretrained=False), cnn_path, device)  # build an empty CNN and fill it with the saved weights
        print("[CNN-Clean] loaded from checkpoint")  # tell the user
    else:  # first run
        cnn_clean = load_cnn_clean(device)  # download the ready-made CNN
        save_checkpoint(cnn_clean, cnn_path)  # save it for next time

    vit_needs_ft = False  # does the ViT still need normal training? assume no
    if os.path.exists(vit_path):  # the ViT was already saved earlier
        shell, _ = load_vit_clean(device, args.vit_backbone, pretrained=False)  # build an empty ViT
        vit_clean = load_checkpoint(shell, vit_path, device)  # fill it with the saved weights
        print("[ViT-Clean] loaded from checkpoint")  # tell the user
    else:  # first run
        vit_clean, vit_needs_ft = load_vit_clean(device, args.vit_backbone)  # download the ViT and learn if it needs normal training

    prepare_data(args.data_root, args.drive_dir)  # make sure the dataset is downloaded
    train_loader = get_train_loader(args.batch_size, args.data_root, args.max_train_samples,  # the object that hands out training images in groups
                                    num_workers=args.num_workers)  # number of helper processes

    if vit_needs_ft:  # the new ViT must first learn CIFAR-10 with normal training
        print(f"[ViT-Clean] clean fine-tune on CIFAR-10 for {args.vit_clean_epochs} epochs")  # tell the user
        clean_finetune(vit_clean, train_loader, epochs=args.vit_clean_epochs, device=device,  # normal training, no tricked images
                       max_batches=max_batches)  # batch limit (only used by the quick test)
        save_checkpoint(vit_clean, vit_path)  # save the trained ViT
    sync_to_drive(args.ckpt_dir, args.drive_dir)  # back up the saved models to Google Drive (if used)
    push("chore(train): clean CNN/ViT baseline checkpoints")  # upload to GitHub (if switched on)

    hist_path = os.path.join(args.ckpt_dir, "training_history.json")  # the log file of training progress
    adv_models = {}  # will hold the two defended models
    for tag_name, clean, method, label in (("cnn_adv", cnn_clean, args.cnn_method, "CNN-AdvTrained"),  # first: (file name, normal model, attack to train with, display name) for the CNN
                                           ("vit_adv", vit_clean, args.vit_method, "ViT-AdvTrained")):  # second: the same four things for the ViT
        ckpt = os.path.join(args.ckpt_dir, f"{tag_name}.pth")  # file of the defended model's weights
        model = copy.deepcopy(clean)  # make a full separate copy, so the normal model stays unchanged
        if (os.path.exists(ckpt) and not args.force_retrain  # a saved defended model exists and we were not told to retrain
                and is_training_complete(hist_path, tag_name, args.epochs)):  # and the log says its training finished
            load_checkpoint(model, ckpt, device)  # just load it
            print(f"[{label}] completed checkpoint found -> skipping training")  # tell the user
        else:  # we need to train it
            print(f"[{label}] adversarial fine-tuning: {args.epochs} epochs, {method}, eps={args.eps}/255")  # tell the user the settings
            if args.force_retrain:  # start again from zero
                for f in (ckpt, os.path.join(args.ckpt_dir, f"{tag_name}_state.pt")):  # the saved model and the "continue later" file
                    if os.path.exists(f):  # the file exists
                        os.remove(f)  # delete it
            adversarial_finetune(model, train_loader, epochs=args.epochs, eps=eps_train, method=method,  # train the copy using normal AND tricked images
                                 lr=args.lr, device=device, tag=tag_name, ckpt_dir=args.ckpt_dir,  # learning rate, GPU/CPU, name, folder for saved files
                                 drive_dir=args.drive_dir, auto_push_every=5 if args.auto_push else 0,  # Drive backup folder; upload to GitHub every 5 epochs if switched on
                                 max_batches=max_batches)  # batch limit (only used by the quick test)
            push(f"chore(train): {label} {args.epochs}-epoch adversarial fine-tuning finished")  # upload to GitHub (if switched on)
        adv_models[label] = model.eval()  # keep this defended model, in test mode

    models = {"CNN-Clean": cnn_clean, "ViT-Clean": vit_clean, **adv_models}  # all four models together: 2 normal + 2 defended
    for m in models.values():  # go through the four models
        for prm in m.parameters():  # go through every learnable number in the model
            prm.requires_grad_(False)  # freeze them: attacks only need to change the image, not the model (saves memory and time)

    X, Y = get_eval_tensors(args.eval_samples, args.data_root)  # the fixed test images X and their true labels Y
    print(f"Evaluation subset: {len(X)} stratified test images (seed {SEED})")  # tell the user how many test images are used
    sweep = evaluate_epsilon_sweep(models, X, Y, device, pgd_steps=pgd_steps)  # test every model against every attack at every strength
    transfer = evaluate_transferability(models, X, Y, device, eps=8 / 255, pgd_steps=pgd_steps)  # test whether tricks made on one model fool the others
    note = (f"Quick-test smoke run only (N={len(X)}, {pgd_steps} PGD steps): not valid results."  # note text for a quick test
            if quick else f"N={len(X)} stratified CIFAR-10 test images, seed {SEED}, PGD-{pgd_steps}.")  # note text for a real run
    export_results(sweep, transfer, args.out_dir, note)  # save the result tables as files

    make_all_plots(sweep, transfer, models, X, Y, device, args.out_dir, pgd_steps=pgd_steps, tag=tag)  # draw and save the four graphs

    info = {"mode": "quick-test" if quick else "full-run", "device": str(device), "epochs": args.epochs,  # a summary of this run: kind of run, device, epochs
            "eval_samples": len(X), "pgd_eval_steps": pgd_steps, "cnn_method": args.cnn_method,  # number of test images, attack steps, CNN training attack
            "vit_method": args.vit_method, "vit_backbone": args.vit_backbone,  # ViT training attack, which ViT
            "clean_acc_full_subset": {k: accuracy(m, X, Y, device) for k, m in models.items()},  # normal accuracy of each model
            "total_minutes": round((time.time() - t_start) / 60, 2)}  # total time in minutes
    save_json(info, os.path.join(args.out_dir, "run_info.json"))  # save the summary to a file
    print(json.dumps(info, indent=2))  # show the summary on screen
    push("results: evaluation tables and plots")  # upload results to GitHub (if switched on)
    print(f"Done in {info['total_minutes']} min. Outputs in {args.out_dir}/")  # tell the user it is finished


if __name__ == "__main__":  # true only when this file is run directly (python main.py)
    main()  # start the program

## Sanity checks

In [ ]:
from src.models import sanity_check  # the quick model check
from src.attacks import self_test  # the quick attack check
sanity_check()  # check both models give 10 scores per image
self_test()  # check the attacks follow their rules

## Optional quick smoke test

In [ ]:
if RUN_QUICK_TEST:  # only if the tiny test was asked for
    # next line runs the tiny test
    !python main.py --quick-test

## Full run (train + evaluate + plots)

In [ ]:
extra = ""  # extra options to add to the command below
if DRIVE_DIR: extra += f" --drive-dir '{DRIVE_DIR}'"  # add the Google Drive backup folder if we have one
if AUTO_PUSH: extra += " --auto-push"  # add automatic GitHub upload if switched on
# next line runs the whole project: loads saved models if they exist, otherwise trains, then tests and draws graphs
!python main.py --full-run --epochs 10 --eval-samples 1000 {extra}

## Results

In [ ]:
import json  # read JSON text files
from IPython.display import Image, Markdown, display  # tools to show pictures and text in the notebook
info = json.load(open("outputs/run_info.json"))  # read the summary of the last run
assert info["mode"] == "full-run", "outputs/ holds a quick-test run, not full-run results. Re-run the training cell."  # stop with an error if the results are only from the tiny test
for f in ["acc_vs_epsilon", "clean_vs_robust_tradeoff", "transferability_heatmap", "adversarial_examples_grid"]:  # the four graphs
    display(Image(filename=f"outputs/{f}.png"))  # show this graph
display(Markdown(open("outputs/results_summary.md", encoding="utf-8").read()))  # show the results tables

In [ ]:
from src.utils import git_auto_push  # the function that uploads to GitHub
if AUTO_PUSH:  # only if uploading is switched on
    git_auto_push("results(final): push complete CIFAR-10 CNN vs ViT robustness metrics and 300-DPI plots [Milestone 7]")  # upload the final results